# SFT, then DPO, on a 360M model on a free T4

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/sft-then-dpo-on-a-colab-t4/sftdpo_v1.ipynb)

Companion to https://profrod.ai/articles/sft-then-dpo-on-a-colab-t4. SmolLM2-360M (the base model, Apache 2.0) learns short arithmetic with worked steps: first by supervised fine-tuning (SFT) with the loss on the reply only, then by DPO on pairs of its own correct and incorrect samples. Every arm is graded by exact match on the same 1,000 test problems, and every gain is compared, item by item, with the base model.

The training loops are plain PyTorch in `sftdpo_run_v1.py`; transformers supplies only the model, the tokenizer and `generate`.

**Before you start:** choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. The default size is estimated at 30 to 50 minutes on a free T4 (from its compute and memory bandwidth; this notebook's receipt is the measurement), well inside a session. Every finished run is saved as it completes; if the session disconnects, run all again and finished runs are skipped. Mount Google Drive in the setup cell to keep results across a runtime reset.

## 1. Setup

In [ ]:
import json
import os
import sys
import time
import urllib.request

T0 = time.time()
SIZE = "default"  # "quick": a 10-minute smoke test; "large": 2,000 test items and 5 seeds, for an L4 or a long session
USE_DRIVE = False  # True: mirror results to Google Drive, so they survive a runtime reset

!pip install -q "transformers==5.18.0"

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/sft-then-dpo-on-a-colab-t4/"
for name in ["sftdpo_task_v1.py", "sftdpo_stats_v1.py", "sftdpo_run_v1.py", "data/revisions-v1.json"]:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
sys.path.insert(0, os.getcwd())

MIRROR = None
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    MIRROR = "/content/drive/MyDrive/sftdpo-v1"

import torch
import sftdpo_run_v1 as R
import sftdpo_task_v1 as T

assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)
lab = R.Lab(SIZE, out="/content/sftdpo-v1", ckpt="/content/sftdpo-ckpt", mirror=MIRROR)
if SIZE == "default":
    # The 1,000 test problems are generated from a seed; this checks you have the same ones.
    assert T.split_digest(lab.splits["test"]) == "b4c582b14939ab59dffb1588a81a742c629ef59883698df30f05358c5a68db77", "test items differ from the article's"
print("model", R.MODEL, "at revision", lab.revision)
print("splits:", {k: len(v) for k, v in lab.splits.items()})
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")

Here is one training example as the model sees it. The labels are `-100` (no loss) for every prompt token, so the loss covers only the reply and its closing `<|im_end|>`. That is the chat-template contract: the same `render` function builds the prompt in training and in evaluation.

In [ ]:
p = lab.splits["sft"][0]
ids, labels = lab.example(p)
print(lab.tok.decode(ids))
print("tokens:", len(ids), "| with loss:", sum(l != -100 for l in labels))

## 2. Base model

Two baselines on the 1,000 test problems: the base model with three worked examples in plain text (`base-fewshot`, the fair one), and the base model zero-shot in the chat template it has never seen (`base-chatml`).

In [ ]:
lab.section_base()
lab.show(arms=("base-fewshot", "base-chatml"))
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")

## 3. SFT

Full fine-tuning, float32 throughout, AdamW, one epoch over 3,000 worked solutions, three seeds (the seed sets the data order). Each checkpoint is saved to `/content/sftdpo-ckpt`.

Why float32 on a T4: the T4 has no fast bfloat16, and float16 is unsafe for this model. Measured in float32, its largest MLP output reaches 48,449 while sampling from left-padded prompts (float16 tops out at 65,504); after four training steps under float16 autocast it reached 72,781, and sampling in float16 failed with NaN probabilities (`results/activations-v1.json`). Why not LoRA: full fine-tuning needs about 16 bytes per parameter for weights, gradients and AdamW state (5.4 GiB for 362M parameters) plus activations for a few thousand tokens, well inside the T4's 15 GiB. The training records include the measured peak.

In [ ]:
lab.section_sft()
s = lab.summarize()
for k, v in s["training"].items():
    if k.startswith("sft"):
        print(f"{k}: {v['steps']} steps, loss {v['firstLoss']} -> {v['finalLoss']}, {v['minutes']:.1f} min, peak {v['peakGiB']:.1f} GiB")
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")

## 4. Evaluate SFT

In [ ]:
lab.section_eval_sft()
lab.show(arms=("base-fewshot", "sft"), comparisons=("sft vs base-fewshot",))
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")

## 5. DPO

From each SFT checkpoint: sample 4 replies at temperature 1 to each of 1,000 fresh problems, keep one (correct, incorrect) pair per problem that has both, and train with the DPO loss, beta 0.1, against the SFT model as the reference. The reference log-probabilities are computed once before training, so only one model sits in memory. The reply token ids are kept exactly as sampled, never re-tokenized from text. Each batch of 16 pairs is processed as two micro-batches of 8 with the gradients accumulated, which gives the same gradient with less memory.

In [ ]:
lab.section_dpo()
s = lab.summarize()
for k, v in s["training"].items():
    if k.startswith("dpo"):
        f = v["final"]
        print(f"{k}: {v['pairStats']['pairs']} pairs (sample accuracy {v['pairStats']['sampleAccuracy']:.1%}), {v['steps']} steps, "
              f"final reward accuracy {f['rewardAccuracy']:.0%}, chosen drift {f['chosenDrift']:+.2f}, rejected drift {f['rejectedDrift']:+.2f} nats, {v['minutes']:.1f} min")
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")

## 6. Evaluate DPO

In [ ]:
lab.section_eval_dpo()
lab.show(arms=("sft", "dpo"), comparisons=("dpo vs sft", "dpo vs base-fewshot"))
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")

## 7. The traps

Three mistakes the course teaches, each measured against SFT with the same seed on the same items:
- `trap-promptloss`: SFT with the loss on every token, the prompt included.
- `trap-sysprompt`: the right SFT model, evaluated through SmolLM2-Instruct's chat template, which inserts a default system prompt the model never saw in training.
- `trap-nogenprompt`: the right SFT model, evaluated without the `<|im_start|>assistant` line that opens its turn (`add_generation_prompt=False`).

In [ ]:
lab.section_trap()
lab.show(
    arms=("sft", "trap-promptloss", "trap-sysprompt", "trap-nogenprompt"),
    comparisons=("trap-promptloss vs sft", "trap-sysprompt vs sft", "trap-nogenprompt vs sft"),
)
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")

## 8. Receipt

The receipt records the GPU, CUDA, the driver, torch and every package version, minutes and peak memory per run, all results and the $0 cost. Download it and send it back: it is the T4 column of the article.

In [ ]:
receipt = lab.receipt()
print(json.dumps(receipt, indent=2))
print(f"[{(time.time() - T0) / 60:.1f} min since setup]")
try:
    from google.colab import files

    files.download("/content/sftdpo-v1/receipts/sftdpo-v1.json")
    files.download("/content/sftdpo-v1/results/runs-v1.jsonl")
except ImportError:
    pass